# 2D Developing Turbulent Pipe Flow — PINN (k-ω)

**Runtime 설정: 상단 메뉴 → Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. 패키지 설치
!pip install -q -U "jax[cuda12]" flax optax ml_collections matplotlib

In [ ]:
# 2. 레포 클론 (fix/review-round-1 브랜치)
!git clone -b fix/review-round-1 https://github.com/powerfulsmooth/RANSwithPirateNet_2D.git

In [ ]:
# 3. GPU 확인
import jax
jax.config.update('jax_default_matmul_precision', 'highest')
print('JAX devices:', jax.devices())

In [ ]:
# 4. 설정 로드
import sys
sys.path.insert(0, '/content/RANSwithPirateNet_2D')
sys.path.insert(0, '/content/RANSwithPirateNet_2D/examples/pipe_2d_komega')

sys.path.insert(0, '/content/RANSwithPirateNet_2D/examples/pipe_2d_komega/configs')
from default import get_config

config = get_config()
config = config.unlock()
config.training.max_steps  = 50000   # T4 기준 약 30~60분
config.training.batch_size = 2048
config.logging.log_every_steps = 500
config.wandb.mode = 'disabled'
config = config.lock()

print(f'Re_tau={config.physics.re_tau}, AR={config.physics.aspect_ratio}, steps={config.training.max_steps}')

In [ ]:
# 5. 학습 실행
import os
os.makedirs('/content/out', exist_ok=True)

import train
model = train.train_and_evaluate(config, '/content/out')

In [ ]:
# 6. 결과 이미지 확인
from IPython.display import Image
Image('/content/out/default/results.png')

In [ ]:
# 7. U+ 프로파일 (outlet) vs 로그 법칙 비교
import numpy as np
import matplotlib.pyplot as plt

eta, yp, Up, Kp, Wp = model.profile_at(model.state.params, xi=1.0)

KAPPA, B = 0.41, 5.0
yv = np.logspace(0.5, np.log10(0.4 * config.physics.re_tau), 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].semilogx(yp, Up, 'b-', lw=2, label='PINN outlet')
axes[0].semilogx(yv, (1/KAPPA)*np.log(yv)+B, 'r--', label='log law')
axes[0].set_xlabel('y+'); axes[0].set_ylabel('U+'); axes[0].legend(); axes[0].set_title('U+ profile')
axes[1].semilogy(eta, np.maximum(Kp, 1e-12), 'g-'); axes[1].set_xlabel('eta'); axes[1].set_title('k+')
axes[2].semilogy(eta, np.maximum(Wp, 1e-12), 'm-'); axes[2].set_xlabel('eta'); axes[2].set_title('omega+')
plt.tight_layout(); plt.show()